# Inspecting realistic data

This notebook will guide you through some simple visualizations you can perform to inspect the data recorded by detectors.

The data was taken from a Loki detector test, and while the data file is realistic, mostly noise was recorded by the detectors.
The techniques for slicing and histogramming the data in different ways are still extremely relevant for early instrument commissioning activities.

In [ ]:
import scipp as sc
import scippneutron as scn
import plopp
import matplotlib.pyplot as plt
import utils

%matplotlib widget

## Load a Loki ESS file

The `utils` module provides a data loader, based on [ScippNexus](https://scipp.github.io/scippnexus/).

In [ ]:
data = utils.load_loki(utils.fetch_data("4-reduction/loki_051657_00000135"))
data

## Inspect bank 0

We begin by taking a first look at the `loki_detector_0` detector bank.

To extract it from the main data structure, we index down into the dict-like structure using a succession of keys:

In [ ]:
bank0 = data["entry"]["instrument"]["loki_detector_0"]["detector_events"]
bank0

### Make a detector image

The data in the detector bank is even data, binned using the 'logical' dimensions `layer, tube, straw, pixel`.
Binned data cannot be plotted directly, it **must be histogrammed first**.

Thankfully, this is easy to do in Scipp, using the `.hist()` method.<br>
The resulting data would still have 4 dimensions, and to make an image we sum along the `layer` and `straw` dimensions,
to leave the remaining `tube` and `pixel` dimensions, which can now be displayed as an image:

In [ ]:
bank0.hist().sum(["layer", "straw"]).plot(cmax=50)

### Plot counts in every tube

It is also possible to further reduce the number of dimensions by also summing along `pixel`,
producing a plot of counts inside each tube.

In [ ]:
bank0.sum(["layer", "straw", "pixel"]).plot()

### Count the number of events

To count all the events in the bank, we can `sum` over all dimensions:

In [ ]:
bank0.sum()  # equivalent to bank0.sum(['layer', 'straw', 'pixel', 'tube'])

### Plot an event_time_offset spectrum

The events contain additional information, such as the neutron `event_time_offset` (which is related to when the neutron arrived at the detector).

To produce a spectrum showing counts as a function of `event_time_offset`,
we first need to **concatenate all the bins together using `.bins.concat()` before we histogram**.<br>
Without this step, we risk creating a very large 5-dimensional data array.

In [ ]:
bank0.bins.concat().hist(event_time_offset=200).plot()

## Place the detector data in a DataGroup

For making the rest of this notebook more user-friendly, we extract the data from the detector banks into a dedicated `DataGroup`.

In [ ]:
dg = sc.DataGroup(
    {
        f"loki_detector_{i}": data["entry"]["instrument"][f"loki_detector_{i}"][
            "detector_events"
        ]
        for i in range(9)
    }
)
dg

## Exercise 1: find the banks that have events

In this detector test, only some of the detector banks were recording events.<br>
Using either summing or plotting, try to find out which out of the 9 detectors banks have events,
and which do not.

**Solution**

In [ ]:
# Method 1: sum the counts in all banks
for key, det in dg.items():
    print(key, det.sum().value)

# Or simply sum directly on the DataGroup
dg.sum()

In [ ]:
# Method 2: plot all banks
fig, axes = plt.subplots(3, 3, figsize=(10, 8))
for (name, da), ax in zip(dg.items(), axes.flatten()):
    da.sum(["layer", "straw"]).plot(cmax=50, title=name, ax=ax)
fig.subplots_adjust(hspace=0.5, wspace=0.5)

## Display layers with the slicer plot

The data reduction packages provide a number of tools for data visualization.<br>
One of them is the `slicer` plot, which allows one to display three-dimensional data as a 2d image and a slider to navigate the 3rd dimension.

So if we sum bank0 along the `straw` dimension only, we can control which `layer` is displayed using the slider:

In [ ]:
plopp.slicer(bank0.sum("straw"), mode="single", cmax=30)

## Masking

An important aspect of data reduction is masking out bad data which degrade the quality of our final result:
faulty tubes, large backgrounds, unwanted signals, etc...

In the section, we show how masks can be added to the detectors, making use of the logical dimensions.

### Mask bad tubes across all layers

In the images of bank0 above, we can see that there is a set of tubes that are recording no events for all 4 layers (tubes numbers 20-23).<br>
The tubes around them show no reason why these tubes should have recorded no events, and point to some fault in the electronics instead.<br>
The tubes need to be masked out, so that the reduction excludes them rather than assuming no neutrons were scattered towards these tubes.

We create a mask with the tube dimension.<br>
We begin with a boolean array where all values are `False` (using `sc.zeros`), and then marking the correct tubes are masked.

In [ ]:
bad_tubes = sc.zeros(sizes={"tube": bank0.sizes["tube"]}, dtype=bool)
bad_tubes.values[20:24] = True

# Assign the mask to bank0 and re-plot
bank0.masks["bad_tubes"] = bad_tubes
bank0.sum(["layer", "straw"]).plot(cmax=50)

The masked tubes now appear as black rather than dark blue.

### Mask a bad tube inside a single layer

Looking at the slicer plot, we also see that some tubes look bad in some layers but not all layers.

By giving the mask more dimensions, we can mask a single tube for a single layer:

In [ ]:
one_tube = sc.zeros(
    sizes={key: bank0.sizes[key] for key in ("layer", "tube")}, dtype=bool
)
# Select which tube we want to mask by slicing with dimension labels
one_tube["layer", 1]["tube", 10].value = True
bank0.masks["one_tube"] = one_tube

plopp.slicer(bank0.sum("straw"), mode="single", cmax=30)

bank0 should now have two masks: `bad_tubes` and `one_tube`.

In [ ]:
bank0

### Exercise 2: mask the ends of the tubes in bank0

The detector images also show that intensity drops towards the left and right ends of the tubes.

This drop in tube efficiency could cause normalization issues, and we wish to mask them out.
Create a mask which should mask the tube ends.

**Hint:** think about which dimension the mask should have.

**Solution:**

In [ ]:
tube_ends = sc.zeros(sizes={key: bank0.sizes[key] for key in ("pixel",)}, dtype=bool)
tube_ends.values[:50] = True
tube_ends.values[460:] = True
bank0.masks["tube_ends"] = tube_ends

plopp.slicer(bank0.sum("straw"), mode="single", cmax=30)

## Instrument view

It is sometimes very useful to plot detectors in 3D, getting a better sense of how things are arranged in physical space.<br>
Scippneutron (`scn`) provides an `instrument_view` function just for this.

To plot bank number 1, we extract it from the DataGroup `dg`, histogram it using `.hist()`, and send it to the instrument view.

In [ ]:
scn.instrument_view(dg["loki_detector_1"].hist(), cmax=10)

### Displaying a single layer

We can send any part of the data to the instrument view, as long as the pixels have a `position` coordinate.<br>
To display only a single layer of tubes in the view, we select the first layer using `['layer', 0]`:

In [ ]:
scn.instrument_view(dg["loki_detector_1"]["layer", 0].hist(), cmax=10)

### Exercise 3: display a pixel slice

Display the same pixel for all tubes of bank 1 in the instrument view (pick a pixel number of your choice).

**Solution:**

In [ ]:
scn.instrument_view(dg["loki_detector_1"]["pixel", 250].hist())

### Multiple banks

We can also send multiple banks to the instrument view, by putting them inside a dict:

In [ ]:
scn.instrument_view({i: dg[f"loki_detector_{i}"].hist() for i in [1, 2, 5, 6]}, cmax=10)

## Plotting profiles

### Individual tubes

We now want to take a closer look at some of the individual tubes in bank number 1.

The instrument view above revealed some high counts towards one edge of some of the tubes.<br>
Here we extract a tube inside the first layer and sum for all straws.<br>
This leaves a one-dimensional intensity as a function of pixel (along the tube length).

In [ ]:
plopp.plot(dg["loki_detector_1"]["layer", 0]["tube", 4].hist().sum("straw"))

### Individual straws

We can also display all straws inside that tube on the same plot,
by once again giving a dict of straws to the plotting function instead of a single entry:

In [ ]:
straws = {
    i: dg["loki_detector_1"]["layer", 0]["tube", 4]["straw", i].hist() for i in range(7)
}

pp.plot(straws)

### event_time_offset distribution per straw

Finally, instead of displaying the intensity along the length of the tube,
we can plot an `event_time_offset` spectrum for each straw.

In [ ]:
straws = {
    i: dg["loki_detector_1"]["layer", 0]["tube", 4]["straw", i]
    .hist(event_time_offset=300)
    .sum("pixel")
    for i in range(7)
}
pp.plot(straws)

We can now see that the large counts in straw 1 (and to lesser extent for straw 0) are not due to events arriving at a specific arrival times.<br>
Instead, the high counts are spread uniformly over the while time range.

## Bonus questions

See if you can find the answers to the following questions by looking at the contents of the file:

- Are there any events in the monitors?
- Can you find the positions of the monitors along the beamline?
- Why does the instrument view fail for `loki_detector_0`?
- How long was this run recorded for?